In [1]:
import pandas as pd
from pathlib import Path

In [29]:
pd.set_option("display.float_format", "{:,.2f}".format)

## Reading

In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "transactions_2026-08-18_17-35-18_0001.csv"

In [3]:
print(f"Exists: {DATA_PATH.exists()}")

Exists: True


## Info

In [4]:
print(f"Size (MB): {DATA_PATH.stat().st_size / 1024 / 1024:.2f}")

df = pd.read_csv(DATA_PATH)

print(f"\nRows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\nMemory (MB): {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f}")

print("\nDtypes:")
print(df.dtypes)

print("\nFirst 3 rows (transposed):")
df.head(3).T

Size (MB): 540.60

Rows: 883,781
Columns: 47

Memory (MB): 1904.70

Dtypes:
actual_worth            float64
area_id                   int64
area_name_ar                str
area_name_en                str
building_name_ar            str
building_name_en            str
has_parking               int64
instance_date               str
master_project_ar           str
master_project_en           str
meter_rent_price        float64
meter_sale_price        float64
nearest_landmark_ar         str
nearest_landmark_en         str
nearest_mall_ar             str
nearest_mall_en             str
nearest_metro_ar            str
nearest_metro_en            str
no_of_parties_role_1    float64
no_of_parties_role_2    float64
no_of_parties_role_3    float64
procedure_area          float64
procedure_id              int64
procedure_name_ar           str
procedure_name_en           str
project_name_ar             str
project_name_en             str
project_number          float64
property_sub_type_ar        

,0,1,2
actual_worth,684999.0,1732880.0,2100000.0
area_id,465,350,506
area_name_ar,وادي الصفا 3,الثنيه الخامسة,اليلايس 1
area_name_en,Wadi Al Safa 3,Al Thanyah Fifth,Al Yelayiss 1
building_name_ar,بن غاطي تيتانيا,الشراع,NaN
building_name_en,BINGHATTI TITANIA,AL SHERA,NaN
has_parking,1,1,0
instance_date,2025-12-16,2016-04-11,2021-03-10
master_project_ar,ماجان,مجمع مركز دبي للسلع المتعددة الرئيسي,NaN
master_project_en,Majan,DMCC Master Community,NaN


## Missing Value Quantification

In [9]:
missing = df.isna().sum().sort_values(ascending=False).reset_index(name="missing_count")

In [14]:
missing['missing_pct'] = 100 * missing['missing_count'] / len(df)
missing = missing[missing["missing_count"] > 0]

In [15]:
print(f"Total rows: {len(df):,}")
print(f"Columns with missing values: {len(missing)}")

Total rows: 883,781
Columns with missing values: 23


In [16]:
missing

,index,missing_count,missing_pct
0,meter_rent_price,865309,97.909889
1,rent_value,865309,97.909889
2,nearest_mall_en,283995,32.134092
3,nearest_mall_ar,283995,32.134092
4,nearest_metro_ar,278521,31.514708
5,nearest_metro_en,278521,31.514708
6,building_name_ar,250366,28.328964
7,building_name_en,250115,28.300563
8,project_number,233503,26.420912
9,project_name_ar,233503,26.420912


## Duplicates

In [22]:
print(f"Total rows: {len(df):,}")

# Exact duplicates across all columns
exact_dups = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dups:,}")
print(f"Exact duplicate percentage: {100 * exact_dups / len(df):.4f}%")

# # Are transaction IDs unique?
unique_txn_ids = df["transaction_id"].nunique()
print(f"\nUnique transaction_id values: {unique_txn_ids:,}")
print(f"Duplicate transaction_id count: {len(df) - unique_txn_ids:,}")

# # Duplicates ignoring transaction_id (same everything else, different ID)
cols_no_id = [c for c in df.columns if c != "transaction_id"]
dup_no_id = df.duplicated(subset=cols_no_id).sum()
print(f"\nDuplicate rows ignoring transaction_id: {dup_no_id:,}")
print(f"Percentage ignoring transaction_id: {100 * dup_no_id / len(df):.4f}%")

# # Show a few exact duplicate groups
if exact_dups > 0:
    print("\nSample exact duplicate rows:")
    print(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(6).T)

Total rows: 883,781
Exact duplicate rows: 0
Exact duplicate percentage: 0.0000%

Unique transaction_id values: 883,781
Duplicate transaction_id count: 0

Duplicate rows ignoring transaction_id: 25,794
Percentage ignoring transaction_id: 2.9186%


**This means, we have identical rows at 2.91% except for transaction_id. We should investigate them.**

In [24]:
dup_mask = df.duplicated(subset=cols_no_id, keep=False)
sample_group = (
    df[dup_mask]
    .groupby(cols_no_id, dropna=False)
    .size()
    .reset_index(name="group_size")
    .sort_values("group_size", ascending=False)
    .head(1)
)

In [25]:
sample_group

,actual_worth,area_id,area_name_ar,area_name_en,building_name_ar,building_name_en,has_parking,instance_date,master_project_ar,master_project_en,...,reg_type_en,reg_type_id,rent_value,rooms_ar,rooms_en,trans_group_ar,trans_group_en,trans_group_id,load_timestamp,group_size
311,171777.0,506,اليلايس 1,Al Yelayiss 1,NaN,NaN,0,2025-01-09,NaN,NaN,...,Existing Properties,1,NaN,NaN,NaN,مبايعات,Sales,1,2026-08-18T17:31:53.000Z,65


In [26]:
print("One duplicate group profile (excluding transaction_id):")
print(sample_group.T)

# Pull the actual rows for that group
group_values = sample_group.iloc[0][cols_no_id].to_dict()
mask = pd.Series(True, index=df.index)
for col, val in group_values.items():
    if pd.isna(val):
        mask &= df[col].isna()
    else:
        mask &= df[col] == val

print(f"\nActual rows in this group: {mask.sum()}")
df[mask][["transaction_id", "instance_date", "actual_worth", "procedure_area", "area_name_en", "property_sub_type_en", "rooms_en"]]

One duplicate group profile (excluding transaction_id):
                                              311
actual_worth                             171777.0
area_id                                       506
area_name_ar                            اليلايس 1
area_name_en                        Al Yelayiss 1
building_name_ar                              NaN
building_name_en                              NaN
has_parking                                     0
instance_date                          2025-01-09
master_project_ar                             NaN
master_project_en                             NaN
meter_rent_price                              NaN
meter_sale_price                          1763.99
nearest_landmark_ar                           NaN
nearest_landmark_en                           NaN
nearest_mall_ar                               NaN
nearest_mall_en                               NaN
nearest_metro_ar                              NaN
nearest_metro_en                            

,transaction_id,instance_date,actual_worth,procedure_area,area_name_en,property_sub_type_en,rooms_en
4905,1-41-2025-1235,2025-01-09,171777.0,97.38,Al Yelayiss 1,NaN,NaN
24320,1-41-2025-1231,2025-01-09,171777.0,97.38,Al Yelayiss 1,NaN,NaN
34319,1-41-2025-1250,2025-01-09,171777.0,97.38,Al Yelayiss 1,NaN,NaN
73396,1-41-2025-1060,2025-01-09,171777.0,97.38,Al Yelayiss 1,NaN,NaN
87563,1-41-2025-1058,2025-01-09,171777.0,97.38,Al Yelayiss 1,NaN,NaN
...,...,...,...,...,...,...,...
840567,1-41-2025-1215,2025-01-09,171777.0,97.38,Al Yelayiss 1,NaN,NaN
841275,1-41-2025-1090,2025-01-09,171777.0,97.38,Al Yelayiss 1,NaN,NaN
841835,1-41-2025-1166,2025-01-09,171777.0,97.38,Al Yelayiss 1,NaN,NaN
860274,1-41-2025-1229,2025-01-09,171777.0,97.38,Al Yelayiss 1,NaN,NaN


**There are multiple duplicate transactions on same date. This seems irregular.**

## Possible Invalid Values

In [27]:
from datetime import datetime

print("=== Numeric impossibilities ===")
print(f"actual_worth <= 0: {(df['actual_worth'] <= 0).sum():,}")
print(f"actual_worth is NaN: {df['actual_worth'].isna().sum():,}")
print(f"procedure_area <= 0: {(df['procedure_area'] <= 0).sum():,}")
print(f"procedure_area is NaN: {df['procedure_area'].isna().sum():,}")
print(f"meter_sale_price <= 0: {(df['meter_sale_price'] <= 0).sum():,}")
print(f"meter_sale_price is NaN: {df['meter_sale_price'].isna().sum():,}")
print(f"rent_value < 0: {(df['rent_value'] < 0).sum():,}")

print("\n=== Categorical/binary impossibilities ===")
print(f"has_parking not in {{0,1}}: {(~df['has_parking'].isin([0, 1])).sum():,}")
print(f"no_of_parties_role_1 < 0: {(df['no_of_parties_role_1'] < 0).sum():,}")
print(f"no_of_parties_role_2 < 0: {(df['no_of_parties_role_2'] < 0).sum():,}")
print(f"no_of_parties_role_3 < 0: {(df['no_of_parties_role_3'] < 0).sum():,}")

print("\n=== Date checks ===")
df_dates = pd.to_datetime(df["instance_date"], errors="coerce")
print(f"instance_date parse failures: {df_dates.isna().sum():,}")
print(f"Dates after today: {(df_dates > datetime.now()).sum():,}")
print(f"Dates before 1970: {(df_dates < '1970-01-01').sum():,}")
print(f"Date range: {df_dates.min()} to {df_dates.max()}")

=== Numeric impossibilities ===
actual_worth <= 0: 0
actual_worth is NaN: 0
procedure_area <= 0: 0
procedure_area is NaN: 0
meter_sale_price <= 0: 3
meter_sale_price is NaN: 0
rent_value < 0: 0

=== Categorical/binary impossibilities ===
has_parking not in {0,1}: 0
no_of_parties_role_1 < 0: 0
no_of_parties_role_2 < 0: 0
no_of_parties_role_3 < 0: 0

=== Date checks ===
instance_date parse failures: 0
Dates after today: 0
Dates before 1970: 3
Date range: 1416-07-02 00:00:00 to 2026-08-17 00:00:00


#### Minor findings

- 3 values with meter_sale_price <= 0
- 3 transactions before 1970 (definitely needs to be removed)

## Suspicious categorical values

In [32]:
categorical_cols = [
    "trans_group_en", "procedure_name_en", "reg_type_en",
    "property_type_en", "property_sub_type_en", "rooms_en",
    "area_name_en", "property_usage_en"
]

for col in categorical_cols:
    print(f"\n=== {col} ===")
    print(df[col].value_counts(dropna=False).head(10))


=== trans_group_en ===
trans_group_en
Sales        677015
Mortgages    173760
Gifts         33006
Name: count, dtype: int64

=== procedure_name_en ===
procedure_name_en
Sell - Pre registration      309401
Sell                         255008
Mortgage Registration        120111
Delayed Sell                  76993
Lease to Own Registration     29768
Grant                         29255
Modify Mortgage               11160
Delayed Mortgage               9870
Sell Development               6449
Development Registration       6309
Name: count, dtype: int64

=== reg_type_en ===
reg_type_en
Existing Properties    562776
Off-Plan Properties    321005
Name: count, dtype: int64

=== property_type_en ===
property_type_en
Unit        633666
Villa       153701
Land         78256
Building     18158
Name: count, dtype: int64

=== property_sub_type_en ===
property_sub_type_en
Flat                  565124
NaN                   173131
Villa                  76842
Office                 36512
Hotel Apartme

## Suspicious distributions & extreme outliers


In [30]:
numeric_cols = ["actual_worth", "procedure_area", "meter_sale_price"]

print("=== Descriptive stats ===")
print(df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

print("\n=== IQR outliers (values beyond 1.5*IQR) ===")
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers):,} outliers ({100*len(outliers)/len(df):.2f}%) | lower={lower:.2f}, upper={upper:.2f}")

print("\n=== Top 5 actual_worth values ===")
print(df.nlargest(5, "actual_worth")[["actual_worth", "procedure_area", "meter_sale_price", "area_name_en", "property_sub_type_en", "instance_date"]])

print("\n=== Top 5 meter_sale_price values ===")
print(df.nlargest(5, "meter_sale_price")[["actual_worth", "procedure_area", "meter_sale_price", "area_name_en", "property_sub_type_en", "instance_date"]])

=== Descriptive stats ===
                      count         mean           std  min         1%  \
actual_worth     883,781.00 3,817,072.58 55,885,640.24 1.00 216,000.00   
procedure_area   883,781.00     1,315.00    373,992.46 0.02      29.72   
meter_sale_price 883,781.00    15,423.90     95,628.12 0.00     504.26   

                         5%        25%          50%          75%          95%  \
actual_worth     362,500.00 750,000.00 1,355,888.00 2,508,608.00 8,198,405.00   
procedure_area        37.12      68.88       109.76       208.02     1,161.10   
meter_sale_price   2,418.31   7,534.73    11,944.02    18,072.48    31,330.99   

                           99%               max  
actual_worth     36,025,010.20 13,786,936,424.00  
procedure_area        3,827.31    342,103,430.80  
meter_sale_price     49,739.23     34,995,777.30  

=== IQR outliers (values beyond 1.5*IQR) ===
actual_worth: 75,229 outliers (8.51%) | lower=-1887912.00, upper=5146520.00
procedure_area: 132,650 ou

### Findings : Many Red Flags

- Actual worth max value: 13.8 Billion
- Max procedure area 342 Million Sq mtr
- Meter Sale Price - 0.09 Physically impossible



## Consistency between related fields

In [31]:
# 1. Does area_id always map to one area_name_en?
area_inconsistent = df.groupby("area_id")["area_name_en"].nunique()
print(f"area_id values with multiple area_name_en: {(area_inconsistent > 1).sum()}")

# 2. Does property_type always match property_sub_type?
type_mismatch = df[df["property_type_en"].notna() & df["property_sub_type_en"].notna()].groupby("property_type_en")["property_sub_type_en"].nunique()
print("\nsub-types per property type:")
print(type_mismatch)

# 3. Does actual_worth roughly equal meter_sale_price * procedure_area?
df_check = df[(df["actual_worth"].notna()) & (df["meter_sale_price"].notna()) & (df["procedure_area"] > 0)]
df_check = df_check.copy()
df_check["implied_worth"] = df_check["meter_sale_price"] * df_check["procedure_area"]
df_check["worth_ratio"] = df_check["actual_worth"] / df_check["implied_worth"]

print("\n=== actual_worth / (meter_sale_price * procedure_area) ===")
print(df_check["worth_ratio"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
print(f"\nRows where ratio < 0.5 or > 2.0: {((df_check['worth_ratio'] < 0.5) | (df_check['worth_ratio'] > 2.0)).sum():,}")

area_id values with multiple area_name_en: 0

sub-types per property type:
property_type_en
Building     1
Unit        18
Villa        1
Name: property_sub_type_en, dtype: int64

=== actual_worth / (meter_sale_price * procedure_area) ===
count   883,781.00
mean           inf
std            NaN
min           0.85
1%            1.00
5%            1.00
25%           1.00
50%           1.00
75%           1.00
95%           1.00
99%           1.00
max            inf
Name: worth_ratio, dtype: float64

Rows where ratio < 0.5 or > 2.0: 3


/Users/mango/code/stake/prediction/.venv/lib/python3.13/site-packages/pandas/core/nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
